In [4]:
# =========================
# IMPORTS & SETUP
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
from PIL import Image
import numpy as np
from torch.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# CLASS MAPPING & AUGMENTATIONS
# =========================
class_mapping = {
    "Corn_Gray_leaf_spot": 0, "Corn_leaf_blight": 1, "Corn_rust_leaf": 2,
    "Tomato_Septoria_leaf_spot": 3, "Tomato_leaf": 4, "Apple_Scab_Leaf": 5,
    "Apple_leaf": 6, "Apple_rust_leaf": 7, "grape_leaf": 8, "grape_leaf_black_rot": 9,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 0,
    "Corn_(maize)___Northern_Leaf_Blight": 1, "Corn_(maize)___Common_rust_": 2,
    "Tomato___Septoria_leaf_spot": 3, "Tomato___healthy": 4,
    "Apple___Apple_scab": 5, "Apple___healthy": 6, "Apple___Cedar_apple_rust": 7,
    "Grape___healthy": 8, "Grape___Black_rot": 9,
}

train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
class PlantDocDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                for img in os.listdir(class_path):
                    self.samples.append(
                        (os.path.join(class_path, img),
                         class_mapping[class_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

plantdoc_root = "segmented/segmented_sam3_new_5"

full_pd_dataset = PlantDocDataset(
    plantdoc_root,
    transform=train_transform
)

indices = list(range(len(full_pd_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

pd_train_subset = Subset(full_pd_dataset, train_idx)

pd_val_dataset = PlantDocDataset(
    plantdoc_root,
    transform=val_transform
)

pd_val_subset = Subset(pd_val_dataset, val_idx)

print("PlantDoc train size:", len(pd_train_subset))
print("PlantDoc val size:", len(pd_val_subset))

class PlantVillageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                valid_extensions = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

                for img in os.listdir(class_path):
                    if img.endswith(valid_extensions):
                        self.samples.append(
                            (os.path.join(class_path, img),
                             class_mapping[class_name])
                        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        if self.transform:
            image = self.transform(image)

        return image, label

plantvillage_root = "leaf_data/plantvillage dataset/color"

pv_dataset = PlantVillageDataset(
    plantvillage_root,
    transform=train_transform
)

print("PlantVillage size:", len(pv_dataset))

joint_train_dataset = ConcatDataset([pv_dataset, pd_train_subset])

train_loader = DataLoader(
    joint_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    pd_val_subset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

num_classes = 10

import timm
import torch.nn as nn

num_classes = 10

model = timm.create_model(
    "vit_small_patch16_dinov3.lvd1689m",
    pretrained=True,
    img_size=224
)

for param in model.parameters():
    param.requires_grad = False

for param in model.blocks[-2:].parameters():
    param.requires_grad = True

in_features = model.num_features
model.head = nn.Linear(in_features, num_classes)

for param in model.head.parameters():
    param.requires_grad = True

model = model.to(device)

print("DINOv3 Small partially unfrozen.")

from collections import Counter

train_labels = [label for _, label in pd_train_subset]
class_counts = Counter(train_labels)

weights = [1.0 / class_counts[i] for i in range(num_classes)]
weights = torch.tensor(weights).float().to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

from torch.amp import GradScaler, autocast

def train_model(model, train_loader, val_loader, epochs=30, patience=10):

    scaler = GradScaler("cuda")
    best_acc = 0
    early_stop = 0

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                with autocast("cuda"):
                    outputs = model(images)

                _, preds = torch.max(outputs,1)

                total += labels.size(0)
                correct += (preds==labels).sum().item()

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_dino_finetuned.pth")
            early_stop = 0
        else:
            early_stop += 1

        if early_stop >= patience:
            print("Early stopping triggered.")
            break

        scheduler.step()

    print("Best Validation Accuracy:", best_acc)

train_model(model, train_loader, val_loader, epochs=30)

Using device: cuda
PlantDoc train size: 746
PlantDoc val size: 187
PlantVillage size: 10205
DINOv3 Small partially unfrozen.
Epoch 1: Loss=1.3352 | Val Acc=0.5936
Epoch 2: Loss=0.4728 | Val Acc=0.7059
Epoch 3: Loss=0.2993 | Val Acc=0.7433
Epoch 4: Loss=0.2434 | Val Acc=0.7594
Epoch 5: Loss=0.2161 | Val Acc=0.7807
Epoch 6: Loss=0.2033 | Val Acc=0.7914
Epoch 7: Loss=0.1946 | Val Acc=0.8075
Epoch 8: Loss=0.1879 | Val Acc=0.8021
Epoch 9: Loss=0.1837 | Val Acc=0.8075
Epoch 10: Loss=0.1783 | Val Acc=0.8182
Epoch 11: Loss=0.1765 | Val Acc=0.8289
Epoch 12: Loss=0.1727 | Val Acc=0.8182
Epoch 13: Loss=0.1704 | Val Acc=0.8235
Epoch 14: Loss=0.1688 | Val Acc=0.8342
Epoch 15: Loss=0.1674 | Val Acc=0.8235
Epoch 16: Loss=0.1651 | Val Acc=0.8342
Epoch 17: Loss=0.1634 | Val Acc=0.8342
Epoch 18: Loss=0.1623 | Val Acc=0.8342
Epoch 19: Loss=0.1621 | Val Acc=0.8342
Epoch 20: Loss=0.1607 | Val Acc=0.8342
Epoch 21: Loss=0.1607 | Val Acc=0.8396
Epoch 22: Loss=0.1600 | Val Acc=0.8449
Epoch 23: Loss=0.1596 | Va

In [5]:
test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

plantdoc_test_root = "plantdoc_test2"

pd_test_dataset = PlantDocDataset(
    plantdoc_test_root,
    transform=test_transform
)

test_loader = DataLoader(
    pd_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("PlantDoc Test size:", len(pd_test_dataset))

import timm
import torch.nn as nn

num_classes = 10

model = timm.create_model(
    "vit_small_patch16_dinov3.lvd1689m",
    pretrained=False,
    img_size=224
)

in_features = model.num_features
model.head = nn.Linear(in_features, num_classes)

model.load_state_dict(
    torch.load("best_dino_finetuned.pth", map_location=device)
)

model = model.to(device)
model.eval()

print("Best DINO model loaded successfully.")

from sklearn.metrics import confusion_matrix, classification_report

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print("\n==============================")
print("DINOv2 Test Accuracy:", test_acc)
print("==============================")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

class_names = [
    "Corn_Gray_leaf_spot",
    "Corn_leaf_blight",
    "Corn_rust_leaf",
    "Tomato_Septoria_leaf_spot",
    "Tomato_leaf",
    "Apple_Scab_Leaf",
    "Apple_leaf",
    "Apple_rust_leaf",
    "grape_leaf",
    "grape_leaf_black_rot",
]

print("\nPer Class Accuracy:")

for i, class_name in enumerate(class_names):
    class_total = cm[i].sum()
    class_correct = cm[i][i]

    acc = class_correct / class_total if class_total > 0 else 0
    print(f"{class_name}: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

PlantDoc Test size: 177
Best DINO model loaded successfully.

DINOv2 Test Accuracy: 0.8587570621468926

Confusion Matrix:
[[ 9  5  1  0  0  0  0  0  0  0]
 [ 8 13  0  0  0  0  0  0  0  0]
 [ 0  1 20  0  0  0  0  0  0  0]
 [ 0  0  0 22  0  0  0  0  0  0]
 [ 0  0  0  1 13  0  0  0  0  0]
 [ 0  0  0  0  0 11  2  2  0  0]
 [ 0  0  0  0  1  0 12  1  0  0]
 [ 0  0  0  1  0  0  1 16  0  0]
 [ 0  0  0  0  0  0  0  0 18  0]
 [ 0  0  0  0  0  0  0  1  0 18]]

Per Class Accuracy:
Corn_Gray_leaf_spot: 0.6000
Corn_leaf_blight: 0.6190
Corn_rust_leaf: 0.9524
Tomato_Septoria_leaf_spot: 1.0000
Tomato_leaf: 0.9286
Apple_Scab_Leaf: 0.7333
Apple_leaf: 0.8571
Apple_rust_leaf: 0.8889
grape_leaf: 1.0000
grape_leaf_black_rot: 0.9474

Classification Report:
                           precision    recall  f1-score   support

      Corn_Gray_leaf_spot       0.53      0.60      0.56        15
         Corn_leaf_blight       0.68      0.62      0.65        21
           Corn_rust_leaf       0.95      0.95      0.9